In [16]:
import polars as pl


# SII

In [56]:
COLS_SII_UTILIZADAS = [
    "Tipo Doc",
    "RUT Emisor",
    "Razon Social",
    "Folio",
    "Fecha Docto",
    "Fecha Recepcion",
    "Fecha Reclamo",
    "Fecha Acuse"
    "Monto Exento",
    "Monto Neto",
    "Monto IVA Recuperable",
    "Monto Total",
]

schema = {
    "Nro": pl.Int64,
    "Tipo Doc": pl.Int64,
    "Tipo Compra": pl.Utf8,
    "RUT Proveedor": pl.Utf8,
    "Razon Social": pl.Utf8,
    "Folio": pl.Int64,
    "Fecha Docto": pl.Utf8,
    "Fecha Recepcion": pl.Utf8,
    "Fecha Acuse": pl.Utf8,
    "Monto Exento": pl.Int64,
    "Monto Neto": pl.Int64,
    "Monto IVA Recuperable": pl.Int64,
    "Monto Iva No Recuperable": pl.Int64,
    "Codigo IVA No Rec.": pl.Int64,
    "Monto Total": pl.Int64,
    "Monto Neto Activo Fijo": pl.Int64,
    "IVA Activo Fijo": pl.Int64,
    "IVA uso Comun": pl.Int64,
    "Impto. Sin Derecho a Credito": pl.Int64,
    "IVA No Retenido": pl.Int64,
    "NCE o NDE sobre Fact. de Compra": pl.Int64,
    "Codigo Otro Impuesto": pl.Utf8,
    "Valor Otro Impuesto": pl.Utf8,
    "Tasa Otro Impuesto": pl.Utf8,
}

df = (
    pl.scan_csv("crudos/base_de_datos_facturas/SII/*.csv", separator=";", dtypes=schema)
    .rename({"RUT Proveedor": "RUT Emisor"})
    .with_columns(
        pl.when((pl.col("Tipo Doc") == 61) | (pl.col("Tipo Doc") == 56)).then(
            pl.col(["Monto Exento", "Monto Neto", "Monto IVA Recuperable", "Monto Total"]) * -1
        )
    )
    .with_columns(pl.concat_str(pl.col(["RUT Emisor", "Folio"])).alias("llave_id"))
    .
)

# ACEPTA

In [59]:
df = pl.read_excel("crudos/base_de_datos_facturas/ACEPTA/*.xls")

InvalidXlsxFileException: Invalid xlsx file: crudos/base_de_datos_facturas/ACEPTA/*.xls

# SCI

In [70]:
df = pl.scan_csv(
    "crudos/base_de_datos_facturas/SCI/*.csv", separator=",", dtypes={"Numero Documento": pl.Utf8}
).rename({"Rut Proveedor": "RUT Emisor", "Numero Documento": "Folio"})


# SIGFE REPORTS - Facturas

In [107]:
df = (
    pl.scan_csv(
        "crudos/base_de_datos_facturas/SIGFE/*.csv", separator=",", skip_rows=10
    )
    .filter((pl.col("Folio").is_not_null()))
    .filter((pl.col("Cuenta Contable") != "Cuenta Contable"))
    .with_columns(pl.col("Principal").str.split(" ").arr[0].alias("RUT Emisor"))
)


In [108]:
df.fetch(10)

Cuenta Contable,Principal,,_duplicated_0,Saldo,Tipo Movimiento,Fecha,Folio,Título,Debe,Haber,Saldo Acumulado,Tipo Documento,Número,Origen Transacción,RUT Emisor
str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str
"""21521 C x P Ga…","""0000509-6 SUEL…",null,null,"""0""","""Tipo Flujo""","""31/07/2015""","""111""","""devenga deudor…","""0""","""2.339.038""","""(2.339.038)""","""Documento de N…","""5320""","""Sigfe Transacc…","""0000509-6"""
"""21521 C x P Ga…","""0000509-6 SUEL…",null,null,"""0""","""Tipo Flujo""","""31/07/2015""","""112""","""cancela deudor…","""2.339.038""","""0""","""0""","""Documento de N…","""5320""","""Sigfe Transacc…","""0000509-6"""
"""21521 C x P Ga…","""0000509-6 SUEL…",null,null,"""0""","""Tipo Flujo""","""31/07/2015""","""113""","""Reversa de : c…","""(2.339.038)""","""0""","""(2.339.038)""","""Documento de N…","""5320""","""Sigfe Transacc…","""0000509-6"""
"""21521 C x P Ga…","""0000509-6 SUEL…",null,null,"""0""","""Tipo Flujo""","""31/07/2015""","""114""","""Reversa de: de…","""0""","""(2.339.038)""","""0""","""Documento de N…","""5320""","""Sigfe Transacc…","""0000509-6"""
"""21521 C x P Ga…","""0000509-6 SUEL…",null,null,"""0""","""Tipo Flujo""","""28/10/2015""","""4483""","""Devenga Sueldo…","""0""","""486.342""","""(486.342)""","""Planillas de R…","""1024688""","""Sigfe Transacc…","""0000509-6"""
"""21521 C x P Ga…","""0000509-6 SUEL…",null,null,"""0""","""Tipo Flujo""","""28/10/2015""","""4488""","""2-4569 CANCELA…","""486.342""","""0""","""0""","""Planillas de R…","""1024688""","""Sigfe Transacc…","""0000509-6"""
"""21521 C x P Ga…","""10119717-4 RIO…",null,null,"""0""","""Tipo Flujo""","""30/11/2015""","""6836""","""Devenga Boleta…","""0""","""1.710.000""","""(1.710.000)""","""Boleta de Hono…","""17""","""Sigfe Transacc…","""10119717-4"""
"""21521 C x P Ga…","""10119717-4 RIO…",null,null,"""0""","""Tipo Flujo""","""30/11/2015""","""6837""","""2-5571 CANCELA…","""1.710.000""","""0""","""0""","""Boleta de Hono…","""17""","""Sigfe Transacc…","""10119717-4"""
"""21521 C x P Ga…","""10119717-4 RIO…",null,null,"""0""","""Tipo Flujo""","""28/12/2015""","""7672""","""Devenga Boleta…","""0""","""950.000""","""(950.000)""","""Boleta de Hono…","""19""","""Sigfe Transacc…","""10119717-4"""
